# Preprocessing — Null Handling & Column Cleanup

**Dataset**: Banorte XB (last 6 months)  
**Actions**:
- `CARD_ISSUER`: fill nulls with `"OTHER"`
- `CARD_TYPE`: fill nulls with `"OTHER"`
- `CUSTO_TRANSACAO`: keep nulls (leave as-is)
- Drop columns: `PROVIDER`, `PAYMENT_AMOUNT`

## 1. Setup & Data Loading

In [1]:
from pathlib import Path

import pandas as pd

pd.set_option("display.max_columns", None)
pd.set_option("display.float_format", "{:,.2f}".format)

DATA_DIR = Path("../data/raw/banorte_xb")
N_MONTHS = 6

In [2]:
all_files = sorted(DATA_DIR.rglob("*.parquet"))
files = all_files[-N_MONTHS:]

print(f"Total parquet files available: {len(all_files)}")
print(f"Loading last {N_MONTHS} months:")
for f in files:
    print(f"  {f.relative_to(DATA_DIR)}")

df = pd.concat([pd.read_parquet(f) for f in files], ignore_index=True)
print(f"\nShape: {df.shape[0]:,} rows x {df.shape[1]} columns")

Total parquet files available: 28
Loading last 6 months:
  2025\10.parquet
  2025\11.parquet
  2025\12.parquet
  2026\01.parquet
  2026\02.parquet
  2026\03.parquet

Shape: 7,965,406 rows x 19 columns


## 2. Null State Before Preprocessing

In [4]:
target_cols = ["CARD_ISSUER", "CARD_TYPE", "CUSTO_TRANSACAO"]
drop_cols = ["PROVIDER", "PAYMENT_AMOUNT"]

print("Null counts BEFORE preprocessing:\n")
for col in target_cols + drop_cols:
    if col in df.columns:
        n = df[col].isna().sum()
        pct = n / len(df) * 100
        print(f"  {col:20s} → {n:>10,} nulls ({pct:.2f}%)")
    else:
        print(f"  {col:20s} → column not found")

Null counts BEFORE preprocessing:

  CARD_ISSUER          →     21,962 nulls (0.28%)
  CARD_TYPE            →         15 nulls (0.00%)
  CUSTO_TRANSACAO      →      9,117 nulls (0.11%)
  PROVIDER             →          0 nulls (0.00%)
  PAYMENT_AMOUNT       →          0 nulls (0.00%)


## 3. Apply Preprocessing

- **CARD_ISSUER**: fill nulls → `"OTHER"`
- **CARD_TYPE**: fill nulls → `"OTHER"`
- **CUSTO_TRANSACAO**: leave nulls as-is
- **Drop**: `PROVIDER`, `PAYMENT_AMOUNT`

In [5]:
df["CARD_ISSUER"] = df["CARD_ISSUER"].fillna("OTHER")
df["CARD_TYPE"] = df["CARD_TYPE"].fillna("OTHER")

cols_before = df.shape[1]
df = df.drop(columns=[c for c in drop_cols if c in df.columns])

print(f"Columns dropped: {cols_before} → {df.shape[1]} ({cols_before - df.shape[1]} removed)")
print(f"Remaining shape: {df.shape[0]:,} rows x {df.shape[1]} columns")

Columns dropped: 19 → 17 (2 removed)
Remaining shape: 7,965,406 rows x 17 columns


## 4. Verify Results

In [6]:
print("Null counts AFTER preprocessing:\n")
for col in ["CARD_ISSUER", "CARD_TYPE", "CUSTO_TRANSACAO"]:
    n = df[col].isna().sum()
    pct = n / len(df) * 100
    print(f"  {col:20s} → {n:>10,} nulls ({pct:.2f}%)")

print(f"\nDropped columns still present? {[c for c in drop_cols if c in df.columns] or 'None — confirmed dropped'}")

Null counts AFTER preprocessing:

  CARD_ISSUER          →          0 nulls (0.00%)
  CARD_TYPE            →          0 nulls (0.00%)
  CUSTO_TRANSACAO      →      9,117 nulls (0.11%)

Dropped columns still present? None — confirmed dropped


In [7]:
print("CARD_ISSUER value counts (top 10):")
print(df["CARD_ISSUER"].value_counts().head(10).to_string())
print("CARD_TYPE value counts:")
print(df["CARD_TYPE"].value_counts().to_string())

CARD_ISSUER value counts (top 10):
CARD_ISSUER
BANCOMER         3827459
BANCOPPEL         747474
SANTANDER         583842
BANAMEX           532051
AZTECA            414398
BANORTE           385093
COMPROPAGO        283760
NUBANK            266321
HSBC              190220
MERCADO LIBRE     155930
CARD_TYPE value counts:
CARD_TYPE
DEBIT      6047313
CREDIT     1917601
PREPAID        316
COMBO          161
OTHER           15


In [8]:
# Full null summary after preprocessing
null_after = df.isna().sum()
null_after = null_after[null_after > 0].sort_values(ascending=False)

if null_after.empty:
    print("No remaining nulls in any column.")
else:
    print("Remaining nulls:\n")
    for col, count in null_after.items():
        print(f"  {col:20s} → {count:>10,} ({count / len(df) * 100:.2f}%)")

Remaining nulls:

  CUSTO_TRANSACAO      →      9,117 (0.11%)


In [9]:
print(f"Final columns ({df.shape[1]}):\n")
print(df.dtypes.to_string())

Final columns (17):

CARD_TYPE                 object
FEE_DATE                  object
CARD_ISSUER               object
OPERATION_TYPE            object
AMOUNT_GROSS              object
AMOUNT_INSTALLMENT        object
INSTALLMENTS              object
MERCHANT_NAME             object
CARD_SCHEME               object
CONFIRM_DATE              object
CAMARA_COMPENSACION       object
CUSTO_TRANSACAO           object
FEE_EBANX                 object
AMOUNT_IVA_INSTALLMENT    object
NET_AMOUNT                object
IVA_EBANX                 object
AFILIACION                object


In [10]:
df.head(5)

,CARD_TYPE,FEE_DATE,CARD_ISSUER,OPERATION_TYPE,AMOUNT_GROSS,AMOUNT_INSTALLMENT,INSTALLMENTS,MERCHANT_NAME,CARD_SCHEME,CONFIRM_DATE,CAMARA_COMPENSACION,CUSTO_TRANSACAO,FEE_EBANX,AMOUNT_IVA_INSTALLMENT,NET_AMOUNT,IVA_EBANX,AFILIACION
0,DEBIT,2025-10-14,BANREGIO,payment,129.0000000000,0.0000000000,1,Amazon Audible MX,visa,2025-10-14,PROSA,0.017,2.1930000000,0.000000000000,126.4561200000,0.350880000000,08882978
1,CREDIT,2025-10-13,BANAMEX,payment,352.5800000000,0.0000000000,1,Temu.com,visa,2025-10-13,BANAMEX,0.017,5.9938600000,0.000000000000,345.6271224000,0.959017600000,09267749
2,CREDIT,2025-10-12,NUBANK,payment,132.0000000000,0.0000000000,1,TikTok - 6222,mastercard,2025-10-12,NUBANK,0.017,2.2440000000,0.000000000000,129.3969600000,0.359040000000,08881883
3,CREDIT,2025-10-13,NUBANK,payment,182.3400000000,0.0000000000,1,Temu.com,mastercard,2025-10-13,NUBANK,0.017,3.0997800000,0.000000000000,178.7442552000,0.495964800000,09267749
4,CREDIT,2025-10-10,BANCOMER,payment,112.3800000000,0.0000000000,1,Temu.com,visa,2025-10-10,BANCOMER,0.017,1.9104600000,0.000000000000,110.1638664000,0.305673600000,09267749
